# WikiGraph Agent Demo
## LLM Wiki Philosophy + LightRAG Engine

LLM Wiki의 "지식이 자라는" 철학을 LightRAG의 그래프 엔진 위에서 실현합니다.

1. **INGEST** — 문서 삽입 + 검증
2. **QUERY** — 그래프 즉시 검색 + 쿼리 로그
3. **EVOLVE** — 쿼리 패턴 분석 → 지식 자동 진화
4. **LINT** — 그래프 건강검진

## 1. Setup

In [1]:
import sys, os, shutil
import numpy as np
sys.path.insert(0, '..')
sys.path.insert(0, '.')

from sentence_transformers import SentenceTransformer
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.utils import EmbeddingFunc

LLM_BASE_URL = 'http://222.117.133.162:30010/v1'
LLM_MODEL    = 'qwen-task-pool'
LLM_API_KEY  = 'asdf'
EMBED_MODEL  = 'sentence-transformers/all-MiniLM-L6-v2'
EMBED_DIM    = 384
WORK_DIR     = '/tmp/wikigraph_demo'

print(f'Loading embedding model: {EMBED_MODEL} ...')
embed_model = SentenceTransformer(EMBED_MODEL)
print('Done.')

async def llm_func(prompt, system_prompt=None, history_messages=[], **kwargs):
    return await openai_complete_if_cache(
        LLM_MODEL, prompt,
        system_prompt=system_prompt,
        history_messages=history_messages,
        api_key=LLM_API_KEY,
        base_url=LLM_BASE_URL,
        **kwargs,
    )

async def embed_func(texts):
    return embed_model.encode(texts, normalize_embeddings=True)

print(f'LLM: {LLM_BASE_URL} ({LLM_MODEL})')
print(f'Embedding: {EMBED_MODEL} (local)')

Loading embedding model: sentence-transformers/all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Done.
LLM: http://222.117.133.162:30010/v1 (qwen-task-pool)
Embedding: sentence-transformers/all-MiniLM-L6-v2 (local)


## 2. LightRAG + WikiGraph Agent 초기화

In [2]:
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)

rag = LightRAG(
    working_dir=WORK_DIR,
    llm_model_func=llm_func,
    embedding_func=EmbeddingFunc(
        embedding_dim=EMBED_DIM,
        max_token_size=8192,
        func=embed_func,
    ),
    addon_params={
        'enable_hybrid_search': True,
        'hybrid_search_mode': 'hybrid',
    },
)
await rag.initialize_storages()

from wikigraph.agent import WikiGraphAgent
from wikigraph.config import WikiGraphConfig

config = WikiGraphConfig(working_dir=WORK_DIR)
agent = WikiGraphAgent(rag, config, llm_func=llm_func)

print(f'LightRAG ready: {WORK_DIR}')
print(f'WikiGraph Agent initialized')
print(f'Hybrid Search: {rag._addon_params["enable_hybrid_search"]}')

INFO: Creating working directory /tmp/wikigraph_demo
INFO: [] Created new empty graph file: /tmp/wikigraph_demo/graph_chunk_entity_relation.graphml
INFO:nano-vectordb:Init {'embedding_dim': 384, 'metric': 'cosine', 'storage_file': '/tmp/wikigraph_demo/vdb_entities.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 384, 'metric': 'cosine', 'storage_file': '/tmp/wikigraph_demo/vdb_relationships.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 384, 'metric': 'cosine', 'storage_file': '/tmp/wikigraph_demo/vdb_chunks.json'} 0 data
INFO: Role LLM Configuration (initialized):
INFO:  - extract: None/None, host=None, max_async=4, timeout=240
INFO:  - keyword: None/None, host=None, max_async=4, timeout=240
INFO:  - query: None/None, host=None, max_async=4, timeout=240
INFO:  - vlm: None/None, host=None, max_async=4, timeout=240
INFO: [] Process 148882 KV load full_docs with 0 records
INFO: [] Process 148882 KV load text_chunks with 0 records
INFO: [] Process 148882 KV load full_entit

LightRAG ready: /tmp/wikigraph_demo
WikiGraph Agent initialized
Hybrid Search: True


## 3. INGEST — 문서 삽입

LLM Wiki처럼 문서를 넣으면 자동으로:
- 청킹 → 임베딩 → 엔티티/관계 추출 → 그래프 저장 → BM25 인덱싱

LLM Wiki와 다른 점: **증분 업데이트** — 기존 그래프를 재구축하지 않음

In [3]:
# 작은 문서 2개로 테스트
docs_dir = os.path.join('..', 'docs')
doc_files = ['FrontendBuildGuide.md', 'UV_LOCK_GUIDE.md']

documents = []
for fname in doc_files:
    with open(os.path.join(docs_dir, fname), 'r') as f:
        documents.append(f.read())
    print(f'  {fname}: {len(documents[-1]):,} chars')

print(f'\nIngesting {len(documents)} documents...')
result = await agent.ingest(documents, file_paths=doc_files)
for msg in result['messages']:
    print(f'  {msg}')

INFO: Processing 2 document(s)
INFO: Parsing (native): doc-334d8353604adf3fd859f3cc1ac19276
INFO: Parsing (native): doc-7277e93ea5fd94650d702825b1ed5cab
INFO: Extracting stage 1/2: FrontendBuildGuide.md
INFO: Processing d-id: doc-334d8353604adf3fd859f3cc1ac19276
INFO: Chunking F(legacy): size=1200, split_only=False, overlap=100, doc_id: doc-334d8353604adf3fd859f3cc1ac19276
INFO: Extracting stage 2/2: UV_LOCK_GUIDE.md
INFO: Processing d-id: doc-7277e93ea5fd94650d702825b1ed5cab
INFO: Chunking F(legacy): size=1200, split_only=False, overlap=100, doc_id: doc-7277e93ea5fd94650d702825b1ed5cab
INFO: BM25 index updated: +2 docs → 2 total
INFO: [BM25] Chunks index updated: +2 → 2 total
INFO: BM25 index updated: +2 docs → 4 total
INFO: [BM25] Chunks index updated: +2 → 4 total
INFO: extract LLM func: 4 new workers initialized (Timeouts: Func: 240s, Worker: 480s, Health Check: 495s)


  FrontendBuildGuide.md: 5,535 chars
  UV_LOCK_GUIDE.md: 5,526 chars

Ingesting 2 documents...


ERROR: Failed to extract entities and relationships: C[1/2]: doc-334d8353604adf3fd859f3cc1ac19276-chunk-000: extract LLM func: Worker execution timeout after 480s
ERROR: Traceback (most recent call last):
  File "/home/bwkim_u/harness_pjt/LightRAG/harness_pjt/../lightrag/utils.py", line 2175, in wait_func
    result = await future
             ^^^^^^^^^^^^
lightrag.utils.WorkerTimeoutError: Worker execution timeout after 480s

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/bwkim_u/harness_pjt/LightRAG/harness_pjt/../lightrag/operate.py", line 3747, in _process_with_semaphore
    result = await _process_single_content(chunk)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/bwkim_u/harness_pjt/LightRAG/harness_pjt/../lightrag/operate.py", line 3503, in _process_single_content
    final_result, timestamp = await use_llm_func_with_cache(
                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File 

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

INFO: [Hybrid Search] Chunk retrieval: Vector=0, BM25=4 → RRF merging
INFO: [Hybrid Search] Chunk retrieval mode=hybrid: 4 results after RRF
INFO: Naive query: 4 chunks (chunk_top_k:20 cosine:0.2)
INFO: Final context: 4 chunks


...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

## 4. QUERY — 쿼리 + 자동 평가

쿼리 결과를 자동 평가하고, 품질이 낮으면 EVOLVE를 트리거합니다.
쿼리 로그가 축적되어 나중에 EVOLVE의 입력이 됩니다.

In [4]:
queries = [
    'How to build the frontend WebUI?',
    'What is UV lock file used for?',
    'How to install frontend dependencies?',
    'What build tools does LightRAG use?',
    'How does bun relate to the frontend build process?',
]

for q in queries:
    print(f'\n{"="*60}')
    result = await agent.query(q)
    for msg in result['messages'][-3:]:
        print(f'  {msg}')
    if result.get('evolved'):
        print('  >>> Knowledge evolved!')

print(f'\n--- Agent Stats ---')
stats = agent.stats
print(f'Total queries: {stats["total_queries"]}')
print(f'Tracked entities: {stats["tracked_entities"]}')

INFO: keyword LLM func: 4 new workers initialized (Timeouts: Func: 240s, Worker: 480s, Health Check: 495s)


INFO:  == LLM cache == saving: mix:keywords:3bc8f18fe2313f6ce3b21d1c5f787ec1
INFO: Query nodes: WebUI (top_k:40, cosine:0.2)
INFO: [Hybrid Search] Entity retrieval mode=vector_only: 0 results
INFO: Query edges: frontend, WebUI, build, development, web development (top_k:40, cosine:0.2)
INFO: [Hybrid Search] Relation retrieval mode=vector_only: 0 results
INFO: [Hybrid Search] Chunk retrieval: Vector=0, BM25=4 → RRF merging
INFO: [Hybrid Search] Chunk retrieval mode=hybrid: 4 results after RRF
INFO: Naive query: 4 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 0 entities, 0 relations, 4 vector chunks
INFO: After truncation: 0 entities, 0 relations
INFO: Round-robin merged chunks: 4 -> 4 (deduplicated 0)
INFO: Final context: 0 entities, 0 relations, 4 chunks
INFO: Final chunks S+F/O: C1/1 C1/2 C1/3 C1/4


  QUERY: 'How to build the frontend WebUI?' → 0 entities, 0 relations, 4 chunks
  EVALUATE: quality=0.40, evolve=LOW QUALITY



INFO:  == LLM cache == saving: mix:keywords:2cd65d237a71bfa363895e02465a33cd
INFO: Query nodes: UV (top_k:40, cosine:0.2)
INFO: [Hybrid Search] Entity retrieval mode=vector_only: 0 results
INFO: Query edges: lock file, purpose (top_k:40, cosine:0.2)
INFO: [Hybrid Search] Relation retrieval mode=vector_only: 0 results
INFO: [Hybrid Search] Chunk retrieval: Vector=0, BM25=4 → RRF merging
INFO: [Hybrid Search] Chunk retrieval mode=hybrid: 4 results after RRF
INFO: Naive query: 4 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 0 entities, 0 relations, 4 vector chunks
INFO: After truncation: 0 entities, 0 relations
INFO: Round-robin merged chunks: 4 -> 4 (deduplicated 0)
INFO: Final context: 0 entities, 0 relations, 4 chunks
INFO: Final chunks S+F/O: C1/1 C1/2 C1/3 C1/4


  QUERY: 'What is UV lock file used for?' → 0 entities, 0 relations, 4 chunks
  EVALUATE: quality=0.40, evolve=LOW QUALITY



...

  QUERY: 'How to install frontend dependencies?' → 0 entities, 0 relations, 4 chunks
  EVALUATE: quality=0.40, evolve=LOW QUALITY



INFO:  == LLM cache == saving: mix:keywords:13c1e1be46b39e8caa26c5c41d053dde
INFO: Query nodes: LightRAG (top_k:40, cosine:0.2)
INFO: [Hybrid Search] Entity retrieval mode=vector_only: 0 results
INFO: Query edges: build tools (top_k:40, cosine:0.2)
INFO: [Hybrid Search] Relation retrieval mode=vector_only: 0 results
INFO: [Hybrid Search] Chunk retrieval: Vector=0, BM25=4 → RRF merging
INFO: [Hybrid Search] Chunk retrieval mode=hybrid: 4 results after RRF
INFO: Naive query: 4 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 0 entities, 0 relations, 4 vector chunks
INFO: After truncation: 0 entities, 0 relations
INFO: Round-robin merged chunks: 4 -> 4 (deduplicated 0)
INFO: Final context: 0 entities, 0 relations, 4 chunks
INFO: Final chunks S+F/O: C1/1 C1/2 C1/3 C1/4


  QUERY: 'What build tools does LightRAG use?' → 0 entities, 0 relations, 4 chunks
  EVALUATE: quality=0.40, evolve=LOW QUALITY



INFO:  == LLM cache == saving: mix:keywords:fb50c111a2db1c6b5056ea45aea438d4
INFO: Query nodes: bun (top_k:40, cosine:0.2)
INFO: [Hybrid Search] Entity retrieval mode=vector_only: 0 results
INFO: Query edges: frontend build process, relationship, integration (top_k:40, cosine:0.2)
INFO: [Hybrid Search] Relation retrieval mode=vector_only: 0 results
INFO: [Hybrid Search] Chunk retrieval: Vector=0, BM25=4 → RRF merging
INFO: [Hybrid Search] Chunk retrieval mode=hybrid: 4 results after RRF
INFO: Naive query: 4 chunks (chunk_top_k:20 cosine:0.2)
INFO: Raw search results: 0 entities, 0 relations, 4 vector chunks
INFO: After truncation: 0 entities, 0 relations
INFO: Round-robin merged chunks: 4 -> 4 (deduplicated 0)
INFO: Final context: 0 entities, 0 relations, 4 chunks
INFO: Final chunks S+F/O: C1/1 C1/2 C1/3 C1/4


  QUERY: 'How does bun relate to the frontend build process?' → 0 entities, 0 relations, 4 chunks
  EVALUATE: quality=0.40, evolve=LOW QUALITY

--- Agent Stats ---
Total queries: 5
Tracked entities: 0


## 5. EVOLVE — 지식 진화

쿼리 로그를 분석하여:
1. 자주 함께 검색되는 엔티티 → 새 관계 생성
2. 실패한 쿼리 → 지식 갭 채우기
3. 멀티홉 경로 → 단축 관계 생성

In [5]:
print('Running EVOLVE...')
result = await agent.evolve()
for msg in result['messages'][-10:]:
    print(f'  {msg}')
print(f'\nMutations applied: {result["applied"]}')

Running EVOLVE...

Mutations applied: 0


## 6. LINT — 그래프 건강검진

LLM Wiki는 전체 wiki를 LLM으로 스캔해야 하지만,
WikiGraph는 **그래프 알고리즘**으로 0토큰 탐지합니다.

In [6]:
print('Running LINT...')
result = await agent.lint()
for msg in result['messages'][-10:]:
    print(f'  {msg}')

if result['findings']:
    print(f'\n--- Findings ({len(result["findings"])}) ---')
    for f in result['findings'][:10]:
        print(f'  [{f["severity"]}] {f["finding_type"]}: {f["entity_name"]}')
        print(f'    {f["details"]}')
        print(f'    Action: {f["suggested_action"]}')

Running LINT...
  LINT: scanning 0 entities
  LINT: no issues found


## 7. 그래프 시각화

In [7]:
import networkx as nx

G = nx.read_graphml(os.path.join(WORK_DIR, 'graph_chunk_entity_relation.graphml'))
print(f'Knowledge Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

# Show evolved edges (source_id contains 'wikigraph_evolve')
evolved_edges = [(u, v) for u, v, d in G.edges(data=True) if 'wikigraph_evolve' in d.get('source_id', '')]
print(f'Evolved edges: {len(evolved_edges)}')
for u, v in evolved_edges:
    print(f'  {u} → {v} (auto-generated by EVOLVE)')

print(f'\nTop entities by degree:')
degrees = sorted(G.degree(), key=lambda x: x[1], reverse=True)
for name, deg in degrees[:10]:
    print(f'  {name:30s} degree={deg}')

FileNotFoundError: [Errno 2] No such file or directory: '/tmp/wikigraph_demo/graph_chunk_entity_relation.graphml'

## 8. Cleanup

In [ ]:
await rag.finalize_storages()
print('Done.')

## Summary

| 차원 | LLM Wiki v2 | WikiGraph Agent |
|---|---|---|
| **스케일** | ~1000페이지 | 수만 노드 |
| **업데이트** | 페이지 15개 재작성 | 노드/엣지 증분 |
| **검색** | 풀컨텍스트 로딩 | 그래프+벡터+BM25 |
| **오류** | 전파됨 | 원본 보존+검증 |
| **건강검진** | LLM 전체 스캔 | 그래프 알고리즘 |
| **지식 진화** | LLM 재작성 | 쿼리 패턴 기반 자동 |

```
WikiGraph = LLM Wiki의 철학 + LightRAG의 엔진
         = 스케일러블하게 자라는 지식 그래프
```